# ONNX for Natural Language Processing — Deep Dive

This notebook provides an in-depth exploration of deploying NLP models with ONNX Runtime,
covering the mathematical foundations of Transformer architectures, attention mechanisms,
and practical considerations for exporting models like BERT and GPT-2.

## 1. The Transformer Revolution in NLP

The Transformer architecture (Vaswani et al., 2017) replaced recurrent networks with
self-attention, enabling massive parallelism and superior long-range dependency modeling.

```
┌─────────────────────────────────────────────────────────────┐
│                  TRANSFORMER ARCHITECTURE                     │
├─────────────────────────────────────────────────────────────┤
│                                                               │
│   Input Tokens                        Output Tokens           │
│       │                                    ▲                  │
│       ▼                                    │                  │
│  ┌──────────┐                      ┌──────────────┐          │
│  │ Embedding │                      │ Linear + SM  │          │
│  │ + PosEnc  │                      └──────┬───────┘          │
│  └────┬─────┘                              │                  │
│       │                                    │                  │
│       ▼                                    │                  │
│  ┌─────────────┐    ┌─────────────┐  ┌────┴────────┐        │
│  │   Encoder    │───▶│  Cross-Attn  │  │   Decoder   │        │
│  │  (Nx layers) │    │             │  │  (Nx layers) │        │
│  └─────────────┘    └─────────────┘  └─────────────┘        │
│                                                               │
│  Each Layer:                                                  │
│  ┌────────────────────────────────────┐                      │
│  │ Multi-Head Self-Attention          │                      │
│  │         ▼                          │                      │
│  │ Add & Layer Norm                   │                      │
│  │         ▼                          │                      │
│  │ Feed-Forward Network (FFN)         │                      │
│  │         ▼                          │                      │
│  │ Add & Layer Norm                   │                      │
│  └────────────────────────────────────┘                      │
└─────────────────────────────────────────────────────────────┘
```

### Key Dimensions

| Symbol | Meaning | BERT-base | GPT-2 |
|--------|---------|-----------|--------|
| $d_{model}$ | Hidden dimension | 768 | 768 |
| $h$ | Number of heads | 12 | 12 |
| $d_k = d_{model}/h$ | Key dimension | 64 | 64 |
| $N$ | Number of layers | 12 | 12 |
| $d_{ff}$ | FFN inner dim | 3072 | 3072 |
| $V$ | Vocabulary size | 30522 | 50257 |

## 2. Scaled Dot-Product Attention

The core building block of the Transformer is **Scaled Dot-Product Attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q \in \mathbb{R}^{n \times d_k}$ — Query matrix (what we're looking for)
- $K \in \mathbb{R}^{m \times d_k}$ — Key matrix (what we match against)
- $V \in \mathbb{R}^{m \times d_v}$ — Value matrix (what we retrieve)
- $\sqrt{d_k}$ — Scaling factor to prevent dot products from growing too large

### Why Scale by $\sqrt{d_k}$?

If $q_i$ and $k_j$ are random vectors with zero mean and unit variance, then:

$$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i \cdot k_i) = d_k$$

So $q \cdot k$ has variance $d_k$, and standard deviation $\sqrt{d_k}$.
Without scaling, for large $d_k$ the softmax saturates to one-hot vectors,
leading to vanishing gradients. Dividing by $\sqrt{d_k}$ normalizes variance to 1.

### Attention as Soft Dictionary Lookup

```
Query q_i ──┐
            ├──▶ score(q_i, k_j) = q_i · k_j / √d_k
Key k_j  ───┘
                      │
                      ▼
              softmax over all j
                      │
                      ▼
              α_ij = attention weight
                      │
                      ▼
              output_i = Σ_j α_ij · v_j   (weighted sum of values)
```

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention.
    
    Args:
        Q: Query matrix [batch, heads, seq_len, d_k]
        K: Key matrix [batch, heads, seq_len, d_k]
        V: Value matrix [batch, heads, seq_len, d_v]
        mask: Optional attention mask
    """
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
    
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    attention_weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attention_weights /= attention_weights.sum(axis=-1, keepdims=True)
    
    output = np.matmul(attention_weights, V)
    return output, attention_weights

# Demonstrate with small example
np.random.seed(42)
batch, heads, seq_len, d_k = 1, 1, 4, 8
Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Input shape: Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"Output shape: {output.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(np.round(weights[0, 0], 3))
print(f"Row sums: {weights[0, 0].sum(axis=-1)}")

## 3. Multi-Head Attention

Instead of a single attention function, we project queries, keys, and values $h$ times
with different learned projections, then concatenate:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(h_1, h_2, \ldots, h_h)W^O$$

where each head is:

$$h_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

With projection matrices:
- $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$
- $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$
- $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$
- $W^O \in \mathbb{R}^{hd_v \times d_{model}}$

### Why Multiple Heads?

Each head can learn to attend to different types of relationships:

```
┌─────────────────────────────────────────────────────────┐
│              MULTI-HEAD ATTENTION                         │
├─────────────────────────────────────────────────────────┤
│                                                           │
│  Input X ∈ R^(n × d_model)                               │
│      │                                                    │
│      ├──▶ W_1^Q, W_1^K, W_1^V ──▶ Head 1 (syntactic)    │
│      ├──▶ W_2^Q, W_2^K, W_2^V ──▶ Head 2 (positional)   │
│      ├──▶ W_3^Q, W_3^K, W_3^V ──▶ Head 3 (semantic)     │
│      │         ...                                        │
│      └──▶ W_h^Q, W_h^K, W_h^V ──▶ Head h (coreference)  │
│                                                           │
│      Concat(head_1, ..., head_h) × W^O                   │
│              │                                            │
│              ▼                                            │
│      Output ∈ R^(n × d_model)                            │
└─────────────────────────────────────────────────────────┘
```

### Computational Complexity

Self-attention has $O(n^2 \cdot d)$ complexity where $n$ is sequence length.
This quadratic scaling is why long-document NLP is challenging:

| Sequence Length | Attention FLOPs (d=768) | Memory |
|----------------|------------------------|--------|
| 128 | ~12.6M | ~64KB |
| 512 | ~201M | ~1MB |
| 2048 | ~3.2B | ~16MB |
| 8192 | ~51.5B | ~256MB |

In [ ]:
class MultiHeadAttention:
    """NumPy implementation of Multi-Head Attention for educational purposes."""
    
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Initialize projection weights
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def split_heads(self, x):
        """Reshape [batch, seq, d_model] -> [batch, heads, seq, d_k]"""
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)
    
    def forward(self, X, mask=None):
        batch, seq_len, _ = X.shape
        
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(0, 2, 1, 3)
        attn_output = attn_output.reshape(batch, seq_len, self.d_model)
        
        output = attn_output @ self.W_O
        return output, attn_weights

# Example usage
np.random.seed(0)
d_model, num_heads = 64, 8
mha = MultiHeadAttention(d_model, num_heads)

X = np.random.randn(1, 10, d_model)  # batch=1, seq_len=10
output, weights = mha.forward(X)
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}  (batch, heads, seq, seq)")

## 4. Positional Encoding

Since attention is permutation-invariant, we inject position information:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

This gives each position a unique encoding, and allows the model to learn relative
positions since $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

### ONNX Considerations for Positional Encoding

When exporting to ONNX, positional encodings can be:
1. **Precomputed as constants** — stored in the graph as initializers
2. **Computed dynamically** — necessary for variable-length sequences
3. **Learned embeddings** (BERT-style) — just another embedding lookup

In [ ]:
def positional_encoding(max_len, d_model):
    """Generate sinusoidal positional encodings."""
    pe = np.zeros((max_len, d_model))
    position = np.arange(0, max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = positional_encoding(128, 64)
print(f"Positional encoding shape: {pe.shape}")
print(f"\nFirst 4 positions, first 8 dimensions:")
print(np.round(pe[:4, :8], 4))
print(f"\nDot product between adjacent positions: {np.dot(pe[0], pe[1]):.4f}")
print(f"Dot product between distant positions:  {np.dot(pe[0], pe[64]):.4f}")

## 5. Feed-Forward Network (FFN)

Each Transformer layer contains a position-wise feed-forward network:

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

Where:
- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$ expands the dimension (typically $d_{ff} = 4 \cdot d_{model}$)
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$ projects back down

The GELU activation used in BERT/GPT:

$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

### ONNX Operator Mapping

```
PyTorch Layer          ONNX Operators
─────────────          ──────────────
nn.Linear(W1)    →     MatMul + Add
nn.GELU()        →     Gelu (opset 20) or Erf-based subgraph
nn.Linear(W2)    →     MatMul + Add
nn.LayerNorm()   →     ReduceMean + Sub + Pow + ReduceMean + Add + Sqrt + Div + Mul + Add
                       (or fused LayerNormalization op)
```

## 6. BERT Architecture for ONNX Export

BERT (Bidirectional Encoder Representations from Transformers) uses only the
encoder stack with bidirectional attention:

```
┌─────────────────────────────────────────────────────┐
│                    BERT MODEL                         │
├─────────────────────────────────────────────────────┤
│                                                       │
│  Input: [CLS] token_1 token_2 ... token_n [SEP]      │
│           │                                           │
│           ▼                                           │
│  ┌─────────────────────────────────┐                 │
│  │ Token Embedding (30522 × 768)   │                 │
│  │ + Position Embedding (512 × 768)│                 │
│  │ + Segment Embedding (2 × 768)   │                 │
│  └────────────┬────────────────────┘                 │
│               │                                       │
│               ▼                                       │
│  ┌─────────────────────────────────┐                 │
│  │      Layer Norm + Dropout       │                 │
│  └────────────┬────────────────────┘                 │
│               │                                       │
│               ▼  (× 12 layers)                       │
│  ┌─────────────────────────────────┐                 │
│  │  Multi-Head Self-Attention      │                 │
│  │  + Residual + LayerNorm         │                 │
│  │  Feed-Forward Network           │                 │
│  │  + Residual + LayerNorm         │                 │
│  └────────────┬────────────────────┘                 │
│               │                                       │
│               ▼                                       │
│  [CLS] → Pooler → Classification                    │
│  All tokens → Token-level tasks                      │
└─────────────────────────────────────────────────────┘
```

### BERT ONNX Export Considerations

1. **Dynamic Axes**: Sequence length and batch size must be dynamic
2. **Attention Mask**: Required input for padding handling
3. **Token Type IDs**: Needed for sentence-pair tasks
4. **Opset Version**: Minimum opset 11 recommended, opset 14+ for better optimization

In [ ]:
import torch
import torch.nn as nn

class SimplifiedBERT(nn.Module):
    """Minimal BERT-like model for demonstrating ONNX export."""
    
    def __init__(self, vocab_size=30522, hidden_size=768, num_layers=2,
                 num_heads=12, max_seq_len=512):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_seq_len, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            activation='gelu',
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pooler = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, input_ids, attention_mask):
        seq_len = input_ids.shape[1]
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.embeddings(input_ids) + self.position_embeddings(position_ids)
        x = self.layer_norm(x)
        
        # Convert attention mask for TransformerEncoder
        src_key_padding_mask = (attention_mask == 0)
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        
        # Pool [CLS] token
        pooled = torch.tanh(self.pooler(x[:, 0, :]))
        return x, pooled

model = SimplifiedBERT(num_layers=2, hidden_size=256, num_heads=8)
model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Export to ONNX with dynamic axes
import os
os.makedirs('outputs', exist_ok=True)

dummy_input_ids = torch.randint(0, 1000, (1, 32))
dummy_attention_mask = torch.ones(1, 32, dtype=torch.long)

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    'outputs/bert_model.onnx',
    input_names=['input_ids', 'attention_mask'],
    output_names=['last_hidden_state', 'pooler_output'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'last_hidden_state': {0: 'batch_size', 1: 'sequence_length'},
        'pooler_output': {0: 'batch_size'}
    },
    opset_version=14,
    do_constant_folding=True
)

file_size = os.path.getsize('outputs/bert_model.onnx') / (1024 * 1024)
print(f"Exported ONNX model size: {file_size:.2f} MB")
print(f"Dynamic axes configured for batch_size and sequence_length")

## 7. GPT-2 and Autoregressive Models

GPT-2 uses a decoder-only architecture with **causal (masked) attention**:

$$\text{CausalAttn}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

Where the causal mask $M$ is:

$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}$$

This ensures each position can only attend to previous positions.

### KV-Cache for Efficient Generation

During autoregressive generation, we cache Key and Value matrices:

```
Step 1: Input "The"      → Compute K₁, V₁, cache them
Step 2: Input "cat"      → Compute K₂, V₂, cache. Attend to [K₁,K₂], [V₁,V₂]
Step 3: Input "sat"      → Compute K₃, V₃, cache. Attend to [K₁,K₂,K₃], [V₁,V₂,V₃]
...

Without KV-Cache:  O(n²) per token, O(n³) total for n tokens
With KV-Cache:     O(n) per token,  O(n²) total for n tokens
```

### KV-Cache Memory Formula

$$\text{KV-Cache Memory} = 2 \cdot n_{layers} \cdot n_{heads} \cdot d_{head} \cdot \text{seq\_len} \cdot \text{batch} \cdot \text{bytes}$$

For GPT-2 (12 layers, 12 heads, d_head=64, FP16):
- Seq_len=1024, batch=1: $2 \times 12 \times 12 \times 64 \times 1024 \times 1 \times 2 = 37.7$ MB
- Seq_len=2048, batch=8: $2 \times 12 \times 12 \times 64 \times 2048 \times 8 \times 2 = 603$ MB

In [ ]:
class GPT2Block(nn.Module):
    """Simplified GPT-2 block with KV-cache support."""
    
    def __init__(self, d_model=256, num_heads=8):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
    
    def forward(self, x, past_key_value=None):
        residual = x
        x = self.ln1(x)
        
        # Causal mask
        seq_len = x.shape[1]
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device),
            diagonal=1
        )
        
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = residual + attn_out
        
        residual = x
        x = residual + self.ffn(self.ln2(x))
        return x

class SimpleGPT2(nn.Module):
    """Minimal GPT-2 for ONNX export demonstration."""
    
    def __init__(self, vocab_size=50257, d_model=256, num_layers=4, num_heads=8, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([GPT2Block(d_model, num_heads) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, input_ids):
        seq_len = input_ids.shape[1]
        pos = torch.arange(0, seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

gpt2 = SimpleGPT2(num_layers=2, d_model=128, num_heads=4)
gpt2.eval()
print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2.parameters()):,}")

# Test forward pass
test_input = torch.randint(0, 1000, (1, 16))
with torch.no_grad():
    logits = gpt2(test_input)
print(f"Input: {test_input.shape} → Logits: {logits.shape}")

## 8. Dynamic Sequence Length Handling

NLP models must handle variable-length inputs. ONNX supports this via dynamic axes:

```
┌─────────────────────────────────────────────────────────────────┐
│          DYNAMIC SHAPE HANDLING IN ONNX                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  Static Export:   input_ids: [1, 128]     ← Fixed, inflexible    │
│  Dynamic Export:  input_ids: [batch, seq]  ← Flexible shapes     │
│                                                                   │
│  ONNX Graph:                                                      │
│  ┌──────────────────────────────┐                                │
│  │ input: float[batch, seq]     │  ← Symbolic dimensions         │
│  │        │                     │                                │
│  │        ▼                     │                                │
│  │ Shape → Gather → Unsqueeze   │  ← Dynamic shape inference    │
│  │        │                     │                                │
│  │        ▼                     │                                │
│  │ Reshape/Expand with          │                                │
│  │ symbolic dims                │                                │
│  └──────────────────────────────┘                                │
│                                                                   │
│  Runtime Binding:                                                 │
│  Session.run(input_ids=[1,32])   ← Works!                        │
│  Session.run(input_ids=[4,256])  ← Also works!                   │
│  Session.run(input_ids=[8,512])  ← Also works!                   │
└─────────────────────────────────────────────────────────────────┘
```

### Padding Strategies for Batched Inference

$$\text{Efficiency} = \frac{\sum_{i} \text{actual\_len}_i}{\text{batch\_size} \times \max(\text{len}_i)}$$

For high efficiency, group similar-length sequences together (bucketing).

In [ ]:
# Export GPT-2 with dynamic axes
dummy_input = torch.randint(0, 1000, (1, 16))

torch.onnx.export(
    gpt2,
    dummy_input,
    'outputs/gpt2_model.onnx',
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size', 1: 'sequence_length'}
    },
    opset_version=14
)

file_size = os.path.getsize('outputs/gpt2_model.onnx') / (1024 * 1024)
print(f"Exported GPT-2 ONNX model: {file_size:.2f} MB")

# Validate with different sequence lengths
import onnxruntime as ort

session = ort.InferenceSession('outputs/gpt2_model.onnx')
print(f"\nTesting dynamic sequence lengths:")
for seq_len in [8, 32, 64, 128]:
    test_ids = np.random.randint(0, 1000, (1, seq_len)).astype(np.int64)
    result = session.run(None, {'input_ids': test_ids})
    print(f"  seq_len={seq_len:3d} → output shape: {result[0].shape}")

## 9. Tokenizer Separation

A critical design principle: **separate the tokenizer from the model in ONNX**.

```
┌─────────────────────────────────────────────────────────────┐
│              NLP INFERENCE PIPELINE                           │
├─────────────────────────────────────────────────────────────┤
│                                                               │
│  "Hello world" (raw text)                                     │
│        │                                                      │
│        ▼                                                      │
│  ┌──────────────────────┐                                    │
│  │    TOKENIZER          │  ← CPU, language-specific         │
│  │  (Python/Rust/C++)    │     Not in ONNX graph             │
│  │                       │     Handles: BPE, WordPiece,      │
│  │  • Normalize text     │              SentencePiece        │
│  │  • Split into tokens  │                                   │
│  │  • Map to IDs         │                                   │
│  │  • Add special tokens │                                   │
│  │  • Create attn mask   │                                   │
│  └──────────┬───────────┘                                    │
│             │                                                 │
│             ▼                                                 │
│  input_ids: [101, 7592, 2088, 102]                           │
│  attn_mask: [1, 1, 1, 1]                                     │
│             │                                                 │
│             ▼                                                 │
│  ┌──────────────────────┐                                    │
│  │    ONNX MODEL         │  ← GPU/NPU accelerated           │
│  │  (Transformer)        │     Pure tensor operations        │
│  └──────────┬───────────┘                                    │
│             │                                                 │
│             ▼                                                 │
│  ┌──────────────────────┐                                    │
│  │   POST-PROCESSING     │  ← CPU                           │
│  │  • Decode token IDs   │                                   │
│  │  • Named entities     │                                   │
│  │  • Span extraction    │                                   │
│  └──────────────────────┘                                    │
└─────────────────────────────────────────────────────────────┘
```

### Why Separate?

1. **Tokenizers aren't differentiable** — no gradients needed, no GPU benefit
2. **String operations** aren't supported in ONNX standard ops
3. **Flexibility** — swap tokenizers without re-exporting models
4. **Portability** — use ONNX Runtime Extensions for custom tokenizer ops if needed

In [ ]:
# Demonstrate tokenizer + ONNX model pipeline
# Using a simple tokenizer simulation

class SimpleTokenizer:
    """Simulates WordPiece tokenizer behavior."""
    
    def __init__(self, vocab_size=30522):
        self.vocab_size = vocab_size
        self.cls_token_id = 101
        self.sep_token_id = 102
        self.pad_token_id = 0
    
    def encode(self, text, max_length=128, padding=True):
        # Simulate tokenization with hash-based mapping
        words = text.lower().split()
        token_ids = [self.cls_token_id]
        for word in words:
            token_ids.append(hash(word) % (self.vocab_size - 200) + 200)
        token_ids.append(self.sep_token_id)
        
        attention_mask = [1] * len(token_ids)
        
        if padding and len(token_ids) < max_length:
            pad_len = max_length - len(token_ids)
            token_ids.extend([self.pad_token_id] * pad_len)
            attention_mask.extend([0] * pad_len)
        
        return {
            'input_ids': np.array([token_ids[:max_length]], dtype=np.int64),
            'attention_mask': np.array([attention_mask[:max_length]], dtype=np.int64)
        }

tokenizer = SimpleTokenizer()
encoded = tokenizer.encode("The quick brown fox jumps over the lazy dog", max_length=16)
print(f"Token IDs shape: {encoded['input_ids'].shape}")
print(f"Token IDs: {encoded['input_ids'][0][:12]}...")
print(f"Attention mask: {encoded['attention_mask'][0][:12]}...")

# Run through ONNX model
bert_session = ort.InferenceSession('outputs/bert_model.onnx')
outputs = bert_session.run(None, encoded)
print(f"\nModel output shapes:")
print(f"  Last hidden state: {outputs[0].shape}")
print(f"  Pooler output: {outputs[1].shape}")

## 10. Layer Normalization in ONNX

Layer Normalization is critical in Transformers:

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

Where:
- $\mu = \frac{1}{H}\sum_{i=1}^{H} x_i$ (mean over hidden dimension)
- $\sigma^2 = \frac{1}{H}\sum_{i=1}^{H}(x_i - \mu)^2$ (variance)
- $\gamma, \beta \in \mathbb{R}^H$ are learned parameters

### ONNX Graph Representation

Without fusion (opset < 17):
```
x → ReduceMean → Sub ─────────────────┐
                   │                    │
                   ▼                    ▼
                  Pow(2) → ReduceMean → Add(ε) → Sqrt → Div → Mul(γ) → Add(β)
```

With fusion (optimized):
```
x → LayerNormalization(γ, β, ε) → output
```

The fused operator is 2-3× faster due to reduced memory bandwidth.

In [ ]:
# Demonstrate the difference between fused and unfused LayerNorm
import onnx
from onnx import helper, TensorProto

def count_ops(model_path):
    """Count operator types in an ONNX model."""
    model = onnx.load(model_path)
    op_counts = {}
    for node in model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
    return op_counts

ops = count_ops('outputs/bert_model.onnx')
print("ONNX Operators in BERT model:")
print("-" * 40)
for op, count in sorted(ops.items(), key=lambda x: -x[1])[:15]:
    print(f"  {op:25s} : {count}")

total_ops = sum(ops.values())
print(f"\nTotal operators: {total_ops}")

## 11. Hugging Face Optimum Integration

Hugging Face Optimum provides seamless ONNX export and optimization:

```
┌──────────────────────────────────────────────────────────────┐
│              HUGGING FACE OPTIMUM WORKFLOW                     │
├──────────────────────────────────────────────────────────────┤
│                                                                │
│  transformers model                                            │
│       │                                                        │
│       ▼                                                        │
│  optimum-cli export onnx                                       │
│       │                                                        │
│       ├──▶ model.onnx (FP32, unoptimized)                     │
│       │                                                        │
│       ▼                                                        │
│  ORTOptimizer (graph optimization)                             │
│       │                                                        │
│       ├──▶ model_optimized.onnx                               │
│       │    • Attention fusion                                  │
│       │    • LayerNorm fusion                                  │
│       │    • Gelu fusion                                       │
│       │    • Skip connection fusion                            │
│       │                                                        │
│       ▼                                                        │
│  ORTQuantizer (quantization)                                   │
│       │                                                        │
│       ├──▶ model_quantized.onnx                               │
│       │    • INT8 weights                                      │
│       │    • ~4x smaller                                       │
│       │    • ~2-3x faster                                      │
│       │                                                        │
│       ▼                                                        │
│  ORTModelForXxx (inference)                                    │
│       • Drop-in replacement for AutoModel                     │
│       • Same tokenizer API                                     │
│       • Automatic IO binding                                   │
└──────────────────────────────────────────────────────────────┘
```

### Performance Gains (typical for BERT-base)

| Configuration | Latency (ms) | Throughput | Size |
|---------------|-------------|------------|------|
| PyTorch FP32 (CPU) | 45.2 | 1.0× | 438 MB |
| ONNX FP32 (CPU) | 28.1 | 1.6× | 438 MB |
| ONNX Optimized (CPU) | 19.7 | 2.3× | 438 MB |
| ONNX INT8 (CPU) | 12.3 | 3.7× | 110 MB |
| ONNX FP16 (GPU) | 4.8 | 9.4× | 219 MB |

In [ ]:
# Demonstrate Optimum-style workflow (using ORT directly)
from onnxruntime.transformers import optimizer as ort_optimizer

# Graph optimization for transformer models
print("ONNX Runtime Transformer Optimization Options:")
print("=" * 50)
print("""
Available optimizations for NLP models:

1. Attention Fusion:
   - Fuses Q/K/V projections + attention + output projection
   - Reduces memory bandwidth by combining ops

2. LayerNormalization Fusion:
   - Combines ReduceMean + Sub + Pow + ... into single op
   - Uses optimized kernel

3. Gelu Approximation:
   - Replaces exact Erf-based GELU with tanh approximation
   - GELU(x) ≈ 0.5x(1 + tanh(√(2/π)(x + 0.044715x³)))

4. SkipLayerNormalization Fusion:
   - Combines Add (skip connection) + LayerNorm into one kernel
   - Saves one memory round-trip

5. EmbedLayerNormalization Fusion:
   - Combines token + position + segment embeddings + LayerNorm
   - Single fused kernel for BERT-style inputs
""")

# Show optimization levels
print("\nOptimization Levels:")
print("-" * 50)
levels = {
    0: "No optimization",
    1: "Basic (constant folding, redundant node elimination)",
    2: "Extended (+ complex fusions like attention)",
    99: "All available optimizations"
}
for level, desc in levels.items():
    print(f"  Level {level:2d}: {desc}")

## 12. Attention Mask Patterns

Different NLP tasks require different attention patterns:

### Bidirectional (BERT)
$$A_{ij} = \begin{cases} 1 & \text{if token } j \text{ is not padding} \\ 0 & \text{otherwise} \end{cases}$$

### Causal (GPT)
$$A_{ij} = \begin{cases} 1 & \text{if } j \leq i \text{ and not padding} \\ 0 & \text{otherwise} \end{cases}$$

### Prefix LM (T5 encoder-decoder)
$$A_{ij} = \begin{cases} 1 & \text{if } j \leq |\text{prefix}| \text{ (bidirectional on prefix)} \\ 1 & \text{if } j \leq i \text{ (causal on rest)} \\ 0 & \text{otherwise} \end{cases}$$

```
Bidirectional:    Causal:           Prefix-LM:
┌─────────┐      ┌─────────┐      ┌─────────┐
│ 1 1 1 1 │      │ 1 0 0 0 │      │ 1 1 1 0 │  ← prefix=3
│ 1 1 1 1 │      │ 1 1 0 0 │      │ 1 1 1 0 │
│ 1 1 1 1 │      │ 1 1 1 0 │      │ 1 1 1 0 │
│ 1 1 1 1 │      │ 1 1 1 1 │      │ 1 1 1 1 │
└─────────┘      └─────────┘      └─────────┘
```

In [ ]:
def create_attention_masks(seq_len, mask_type='bidirectional', prefix_len=None):
    """Create different attention mask patterns."""
    if mask_type == 'bidirectional':
        return np.ones((seq_len, seq_len), dtype=np.float32)
    
    elif mask_type == 'causal':
        return np.tril(np.ones((seq_len, seq_len), dtype=np.float32))
    
    elif mask_type == 'prefix_lm':
        mask = np.zeros((seq_len, seq_len), dtype=np.float32)
        mask[:, :prefix_len] = 1  # All can attend to prefix
        mask += np.tril(np.ones((seq_len, seq_len), dtype=np.float32))  # Causal for rest
        return np.clip(mask, 0, 1)
    
    elif mask_type == 'sliding_window':
        mask = np.zeros((seq_len, seq_len), dtype=np.float32)
        window = min(4, seq_len)
        for i in range(seq_len):
            start = max(0, i - window + 1)
            mask[i, start:i+1] = 1
        return mask

print("Attention Mask Patterns (8×8):")
print("=" * 50)
for mtype in ['bidirectional', 'causal', 'prefix_lm', 'sliding_window']:
    kwargs = {'prefix_len': 4} if mtype == 'prefix_lm' else {}
    mask = create_attention_masks(8, mtype, **kwargs)
    print(f"\n{mtype.upper()}:")
    for row in mask:
        print('  ' + ' '.join(['█' if v > 0 else '·' for v in row]))

## 13. NLP Task-Specific ONNX Export Patterns

Different NLP tasks require different model configurations:

```
┌────────────────────────────────────────────────────────────────┐
│                 NLP TASK → ONNX MAPPING                         │
├──────────────────┬─────────────────────────────────────────────┤
│ Task             │ Inputs              → Outputs               │
├──────────────────┼─────────────────────────────────────────────┤
│ Classification   │ input_ids, attn     → logits [B, classes]   │
│ Token Class.     │ input_ids, attn     → logits [B, S, tags]   │
│ QA (extractive)  │ input_ids, attn     → start, end [B, S]     │
│ Seq2Seq          │ enc_ids, dec_ids    → logits [B, T, vocab]  │
│ Generation       │ input_ids, past_kv  → logits, new_kv        │
│ Embedding        │ input_ids, attn     → embeddings [B, S, H]  │
└──────────────────┴─────────────────────────────────────────────┘
```

### Memory Requirements for Common Models

$$\text{Model Memory (FP32)} \approx 4 \times \text{num\_params (in bytes)}$$

$$\text{Activation Memory} \approx 2 \times B \times S \times H \times N_{layers} \times \text{bytes\_per\_element}$$

| Model | Params | FP32 Size | FP16 Size | INT8 Size |
|-------|--------|-----------|-----------|-----------|
| BERT-base | 110M | 440 MB | 220 MB | 110 MB |
| BERT-large | 340M | 1.3 GB | 680 MB | 340 MB |
| GPT-2 | 117M | 468 MB | 234 MB | 117 MB |
| GPT-2 Medium | 345M | 1.4 GB | 700 MB | 345 MB |
| T5-base | 220M | 880 MB | 440 MB | 220 MB |

In [ ]:
# Model size and inference cost calculator

def transformer_flops(seq_len, d_model, num_layers, num_heads, vocab_size, d_ff=None):
    """Estimate FLOPs for a single forward pass of a Transformer."""
    if d_ff is None:
        d_ff = 4 * d_model
    
    flops_per_layer = {
        'qkv_projection': 3 * 2 * seq_len * d_model * d_model,
        'attention_scores': 2 * seq_len * seq_len * d_model,
        'attention_values': 2 * seq_len * seq_len * d_model,
        'output_projection': 2 * seq_len * d_model * d_model,
        'ffn_up': 2 * seq_len * d_model * d_ff,
        'ffn_down': 2 * seq_len * d_ff * d_model,
        'layer_norms': 4 * seq_len * d_model,  # 2 LayerNorms
    }
    
    total_per_layer = sum(flops_per_layer.values())
    embedding_flops = 2 * seq_len * vocab_size  # Lookup (negligible)
    lm_head_flops = 2 * seq_len * d_model * vocab_size
    
    total = num_layers * total_per_layer + embedding_flops + lm_head_flops
    return total, flops_per_layer

# BERT-base analysis
configs = {
    'BERT-base': (512, 768, 12, 12, 30522),
    'GPT-2': (1024, 768, 12, 12, 50257),
    'GPT-2 Medium': (1024, 1024, 24, 16, 50257),
}

print("Transformer FLOPs Analysis")
print("=" * 60)
for name, (seq, d, layers, heads, vocab) in configs.items():
    total, breakdown = transformer_flops(seq, d, layers, heads, vocab)
    print(f"\n{name} (seq_len={seq}):")
    print(f"  Total: {total/1e9:.1f} GFLOPs")
    print(f"  Per layer breakdown:")
    for op, flops in breakdown.items():
        pct = flops / sum(breakdown.values()) * 100
        print(f"    {op:20s}: {flops/1e6:8.1f} MFLOPs ({pct:5.1f}%)")

## 14. ONNX Runtime Execution Providers for NLP

Different hardware targets for NLP inference:

```
┌─────────────────────────────────────────────────────────────────┐
│           EXECUTION PROVIDER SELECTION                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  NLP Model Characteristics:                                       │
│  • Memory-bandwidth bound (large weight matrices)                │
│  • Compute-bound for long sequences (O(n²) attention)            │
│  • Batch size varies (1 for real-time, 64+ for batch)            │
│                                                                   │
│  Provider Selection Guide:                                        │
│                                                                   │
│  ┌─────────────┐  Small batch, latency-critical                  │
│  │     CPU     │  • Single inference, short sequences             │
│  │  (Default)  │  • Edge devices                                  │
│  └─────────────┘  • Best with INT8 quantization                  │
│                                                                   │
│  ┌─────────────┐  Large batch, high throughput                   │
│  │    CUDA     │  • Batch processing                              │
│  │  (GPU)      │  • Long sequences (>512)                         │
│  └─────────────┘  • Best with FP16                               │
│                                                                   │
│  ┌─────────────┐  Maximum GPU performance                        │
│  │  TensorRT   │  • Static or semi-dynamic shapes                │
│  │             │  • Production deployment                         │
│  └─────────────┘  • Requires shape profiling                     │
│                                                                   │
│  ┌─────────────┐  Mobile/Edge NLP                                │
│  │    NNAPI    │  • On-device inference                           │
│  │  CoreML     │  • Quantized models                              │
│  └─────────────┘  • Limited sequence lengths                     │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
import time

# Benchmark ONNX Runtime inference
def benchmark_nlp_inference(session, inputs, num_runs=50, warmup=10):
    """Benchmark NLP model inference with warmup."""
    # Warmup
    for _ in range(warmup):
        session.run(None, inputs)
    
    # Benchmark
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        session.run(None, inputs)
        latencies.append((time.perf_counter() - start) * 1000)
    
    return {
        'mean_ms': np.mean(latencies),
        'p50_ms': np.percentile(latencies, 50),
        'p95_ms': np.percentile(latencies, 95),
        'p99_ms': np.percentile(latencies, 99),
        'std_ms': np.std(latencies)
    }

# Benchmark with different sequence lengths
print("Inference Latency vs Sequence Length")
print("=" * 55)
print(f"{'Seq Len':>8} {'Mean (ms)':>10} {'P50 (ms)':>10} {'P95 (ms)':>10}")
print("-" * 55)

session = ort.InferenceSession('outputs/gpt2_model.onnx')
for seq_len in [16, 32, 64, 128, 256]:
    inputs = {'input_ids': np.random.randint(0, 1000, (1, seq_len)).astype(np.int64)}
    results = benchmark_nlp_inference(session, inputs, num_runs=30, warmup=5)
    print(f"{seq_len:>8} {results['mean_ms']:>10.2f} {results['p50_ms']:>10.2f} {results['p95_ms']:>10.2f}")

print("\n(Latency grows quadratically with sequence length due to attention)")

## 15. Quantization Strategies for NLP Models

Quantization maps FP32 weights/activations to lower precision:

$$x_q = \text{round}\left(\frac{x}{s}\right) + z$$

Where $s$ is scale and $z$ is zero-point. Dequantization:

$$x \approx s \cdot (x_q - z)$$

### NLP-Specific Quantization Challenges

1. **Attention logits** have high dynamic range → sensitive to quantization
2. **LayerNorm** requires FP32 for numerical stability
3. **Embedding layers** benefit less from quantization (lookup, not compute)

### Recommended Strategy for BERT-like Models

```
┌──────────────────────┬───────────────────────────┐
│ Layer Type           │ Quantization              │
├──────────────────────┼───────────────────────────┤
│ Embedding            │ FP32 (keep original)      │
│ Q/K/V Projections    │ INT8 (dynamic quant)      │
│ Attention MatMul     │ Keep FP32 or use FP16     │
│ Output Projection    │ INT8 (dynamic quant)      │
│ FFN Linear layers    │ INT8 (dynamic quant)      │
│ LayerNorm            │ FP32 (keep original)      │
│ Final classifier     │ INT8 or FP32              │
└──────────────────────┴───────────────────────────┘
```

### Accuracy Impact

$$\text{Accuracy Drop} = \frac{\text{Acc}_{FP32} - \text{Acc}_{INT8}}{\text{Acc}_{FP32}} \times 100\%$$

Typical results on GLUE benchmark:
- Dynamic INT8: < 1% accuracy drop
- Static INT8: < 0.5% with proper calibration
- INT4 (GPTQ-style): 1-3% drop but 4× memory reduction

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Apply dynamic quantization to the BERT model
quantize_dynamic(
    'outputs/bert_model.onnx',
    'outputs/bert_model_int8.onnx',
    weight_type=QuantType.QInt8
)

# Compare sizes
fp32_size = os.path.getsize('outputs/bert_model.onnx') / (1024 * 1024)
int8_size = os.path.getsize('outputs/bert_model_int8.onnx') / (1024 * 1024)

print(f"Model Size Comparison:")
print(f"  FP32: {fp32_size:.2f} MB")
print(f"  INT8: {int8_size:.2f} MB")
print(f"  Compression: {fp32_size/int8_size:.2f}×")

# Verify outputs match closely
fp32_session = ort.InferenceSession('outputs/bert_model.onnx')
int8_session = ort.InferenceSession('outputs/bert_model_int8.onnx')

test_input = {
    'input_ids': np.random.randint(0, 1000, (1, 32)).astype(np.int64),
    'attention_mask': np.ones((1, 32), dtype=np.int64)
}

fp32_out = fp32_session.run(None, test_input)[0]
int8_out = int8_session.run(None, test_input)[0]

max_diff = np.max(np.abs(fp32_out - int8_out))
mean_diff = np.mean(np.abs(fp32_out - int8_out))
print(f"\nNumerical Difference (FP32 vs INT8):")
print(f"  Max absolute diff: {max_diff:.6f}")
print(f"  Mean absolute diff: {mean_diff:.6f}")
print(f"  Cosine similarity: {np.dot(fp32_out.flatten(), int8_out.flatten()) / (np.linalg.norm(fp32_out) * np.linalg.norm(int8_out)):.6f}")

## 16. Common ONNX Export Pitfalls for NLP

### Issue 1: Control Flow in Beam Search

Beam search involves dynamic loops that are hard to trace:

$$\text{score}(y_{1:t}) = \sum_{i=1}^{t} \log P(y_i | y_{<i}, x)$$

**Solution**: Export the model without beam search; implement beam search
as a post-processing loop calling the ONNX model iteratively.

### Issue 2: Dynamic Shapes with Attention Cache

KV-cache grows dynamically during generation:
```
Step t:   past_key_values shape = [batch, heads, t-1, d_head]
Step t+1: past_key_values shape = [batch, heads, t, d_head]
```

**Solution**: Use dynamic axes on the sequence dimension of cache tensors.

### Issue 3: Custom Operations

Some NLP operations don't map to standard ONNX ops:
- Rotary Position Embeddings (RoPE)
- Flash Attention
- Mixture of Experts routing

**Solution**: Register custom ONNX ops or decompose into standard ops.

### Issue 4: Numerical Precision

Softmax overflow for large logits:
$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

The $-\max(x)$ trick prevents overflow but must be preserved in ONNX export.

In [ ]:
# Demonstrate autoregressive generation with ONNX model

def greedy_generate(session, input_ids, max_new_tokens=20):
    """Simple greedy decoding using ONNX model."""
    generated = list(input_ids[0])
    
    for _ in range(max_new_tokens):
        current_input = np.array([generated], dtype=np.int64)
        logits = session.run(None, {'input_ids': current_input})[0]
        
        # Get logits for last position
        next_token_logits = logits[0, -1, :]
        next_token = np.argmax(next_token_logits)
        generated.append(int(next_token))
    
    return np.array([generated])

# Generate with our GPT-2 model
prompt = np.array([[100, 200, 300, 400]], dtype=np.int64)
generated = greedy_generate(session, prompt, max_new_tokens=10)
print(f"Prompt tokens: {prompt[0]}")
print(f"Generated tokens: {generated[0]}")
print(f"New tokens: {generated[0][len(prompt[0]):]}") 
print(f"\nNote: Random weights produce random tokens.")
print(f"With a trained model, this would produce coherent text.")

## 17. Production Deployment Patterns

### Batched Inference with Dynamic Batching

```
┌───────────────────────────────────────────────────────────────┐
│              DYNAMIC BATCHING SERVER                            │
├───────────────────────────────────────────────────────────────┤
│                                                                 │
│  Request Queue:                                                 │
│  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐                             │
│  │ R1  │ │ R2  │ │ R3  │ │ R4  │  (arriving requests)        │
│  │s=32 │ │s=64 │ │s=48 │ │s=56 │                             │
│  └──┬──┘ └──┬──┘ └──┬──┘ └──┬──┘                             │
│     │       │       │       │                                  │
│     └───────┴───────┴───────┘                                  │
│                 │                                               │
│                 ▼                                               │
│     ┌───────────────────────┐                                  │
│     │   PAD TO max_len=64   │                                  │
│     │   BATCH together      │                                  │
│     └───────────┬───────────┘                                  │
│                 │                                               │
│                 ▼                                               │
│     ┌───────────────────────┐                                  │
│     │  ONNX Runtime Session │                                  │
│     │  input: [4, 64]       │                                  │
│     └───────────┬───────────┘                                  │
│                 │                                               │
│                 ▼                                               │
│     Unbatch + Return to each client                            │
└───────────────────────────────────────────────────────────────┘
```

### Throughput Formula

$$\text{Throughput} = \frac{\text{batch\_size}}{\text{latency\_per\_batch}}$$

$$\text{Optimal batch size} = \arg\max_B \frac{B}{L(B)} \text{ subject to } L(B) \leq \text{SLA}$$

In [ ]:
# Simulate batched vs single inference throughput

def measure_throughput(session, seq_len, batch_sizes, num_iterations=20):
    """Measure throughput for different batch sizes."""
    results = []
    for bs in batch_sizes:
        inputs = {'input_ids': np.random.randint(0, 1000, (bs, seq_len)).astype(np.int64)}
        
        # Warmup
        for _ in range(5):
            session.run(None, inputs)
        
        # Measure
        start = time.perf_counter()
        for _ in range(num_iterations):
            session.run(None, inputs)
        elapsed = time.perf_counter() - start
        
        latency = elapsed / num_iterations * 1000  # ms
        throughput = bs / (elapsed / num_iterations)  # samples/sec
        results.append((bs, latency, throughput))
    
    return results

batch_sizes = [1, 2, 4, 8, 16]
results = measure_throughput(session, 32, batch_sizes)

print("Batch Size vs Throughput (seq_len=32)")
print("=" * 55)
print(f"{'Batch':>6} {'Latency (ms)':>14} {'Throughput':>14} {'Efficiency':>12}")
print("-" * 55)
base_throughput = results[0][2]
for bs, lat, tp in results:
    efficiency = tp / (base_throughput * bs) * 100
    print(f"{bs:>6} {lat:>14.2f} {tp:>11.1f}/s {efficiency:>10.1f}%")

## 18. Summary and Best Practices

### Key Takeaways for NLP + ONNX

1. **Architecture Understanding**: Know the math behind attention to debug export issues
2. **Dynamic Axes**: Always configure for variable batch and sequence length
3. **Tokenizer Separation**: Keep tokenization outside the ONNX graph
4. **Optimization Pipeline**: Export → Graph Optimize → Quantize → Deploy
5. **KV-Cache**: Essential for efficient autoregressive generation
6. **Precision Selection**: INT8 for CPU, FP16 for GPU, with task-specific tuning

### Performance Checklist

```
□ Export with opset ≥ 14 for best operator support
□ Enable constant folding during export
□ Apply transformer-specific graph optimizations
□ Use dynamic quantization (INT8) for CPU deployment
□ Profile with different sequence lengths and batch sizes
□ Implement sequence bucketing for variable-length inputs
□ Use IO Binding for GPU inference to avoid copies
□ Consider model distillation for extreme latency requirements
```

### Further Reading

- Vaswani et al., "Attention Is All You Need" (2017)
- Devlin et al., "BERT: Pre-training of Deep Bidirectional Transformers" (2018)
- Radford et al., "Language Models are Unsupervised Multitask Learners" (GPT-2, 2019)
- Hugging Face Optimum documentation
- ONNX Runtime Transformer optimization docs

In [ ]:
# Final summary: Complete NLP ONNX workflow
print("""
╔══════════════════════════════════════════════════════════════╗
║         COMPLETE NLP + ONNX WORKFLOW SUMMARY                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. TRAIN (PyTorch/TensorFlow)                               ║
║     └─ Fine-tune BERT/GPT-2 on your task                     ║
║                                                              ║
║  2. EXPORT (torch.onnx.export / Optimum)                     ║
║     ├─ Set dynamic_axes for batch & sequence                 ║
║     ├─ Use opset ≥ 14                                        ║
║     └─ Validate with onnx.checker                            ║
║                                                              ║
║  3. OPTIMIZE (ORT Transformer Optimizer)                     ║
║     ├─ Attention fusion                                      ║
║     ├─ LayerNorm fusion                                      ║
║     └─ Gelu fusion                                           ║
║                                                              ║
║  4. QUANTIZE (Dynamic INT8 for CPU)                          ║
║     ├─ ~4× size reduction                                    ║
║     └─ ~2-3× speedup with <1% accuracy loss                 ║
║                                                              ║
║  5. DEPLOY (ONNX Runtime)                                    ║
║     ├─ CPU: CPUExecutionProvider + INT8                      ║
║     ├─ GPU: CUDAExecutionProvider + FP16                     ║
║     └─ Edge: NNAPI/CoreML providers                          ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")